## handle unique product

In [ ]:
import pandas as pd
from pathlib import Path

In [ ]:
label_path = Path("fashion_compatibility_prediction.txt")
product_ids = set()

with open(label_path, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split()

        # Skip label (first element)
        product_ids.update(parts[1:])

print("Unique products:", len(product_ids))

Unique products: 18604


In [ ]:
records = []

with open("fashion_compatibility_prediction.txt", "r") as f:
    for i, line in enumerate(f):
        parts = line.strip().split()
        records.append({
            "outfit_id": f"O{i+1:04d}",
            "label": int(parts[0]),
            "products": parts[1:]
        })

df = pd.DataFrame(records)
df

,outfit_id,label,products
0,O0001,1,"[119704139_1, 119704139_2, 119704139_3, 119704..."
1,O0002,1,"[119314458_1, 119314458_2, 119314458_3, 119314..."
2,O0003,1,"[147361785_1, 147361785_2, 147361785_3, 147361..."
3,O0004,1,"[148015877_1, 148015877_2, 148015877_3, 148015..."
4,O0005,1,"[192637078_1, 192637078_2, 192637078_3, 192637..."
...,...,...,...
7071,O7072,0,"[184041913_1, 209222683_1, 185007152_3, 207694..."
7072,O7073,0,"[209709393_3, 212653508_4, 189949776_6, 166128..."
7073,O7074,0,"[109684687_6, 104980596_1, 179458125_4, 129142..."
7074,O7075,0,"[215502815_1, 207577310_1, 132234026_1, 127969..."


In [ ]:
import zipfile
import shutil
from pathlib import Path
from tqdm.auto import tqdm

zip_path = "Re-PolyVore.zip"
output_dir = Path("selected_images")

output_dir.mkdir(exist_ok=True)

found_ids = set()

with zipfile.ZipFile(zip_path, "r") as z:

    image_paths = [
        name for name in z.namelist()
        if name.lower().endswith(".jpg")
    ]

    for path in tqdm(image_paths):

        product_id = Path(path).stem

        if product_id in product_ids:

            output_path = output_dir / f"{product_id}.jpg"

            with z.open(path) as source:
                with open(output_path, "wb") as target:
                    shutil.copyfileobj(source, target)

            found_ids.add(product_id)

print("Required products:", len(product_ids))
print("Found images:", len(found_ids))
print("Missing images:", len(product_ids - found_ids))

100%|██████████| 126928/126928 [00:11<00:00, 10592.53it/s]

Required products: 18604
Found images: 8850
Missing images: 9754


In [ ]:
missing_ids = product_ids - found_ids

df_clean = df.copy()

df_clean["products"] = df_clean["products"].apply(
    lambda products: [pid for pid in products if pid not in missing_ids]
)

# Keep outfits with at least 3 products
df_clean = df_clean[
    df_clean["products"].apply(len) >= 3
].reset_index(drop=True)

print("Original outfits:", len(df))
print("Remaining outfits:", len(df_clean))
print("Removed outfits:", len(df) - len(df_clean))
print(df_clean["label"].value_counts())

Original outfits: 7076
Remaining outfits: 4135
Removed outfits: 2941
label
0    2343
1    1792
Name: count, dtype: int64


In [ ]:
import json

df_save = df_clean.copy()
df_save["products"] = df_save["products"].apply(json.dumps)
df_save.to_csv("clean_outfits_compatibility_prediction.csv", index=False)

print("Saved:", len(df_save), "outfits")

Saved: 4135 outfits
